# Merge dataset without meteo
`electricity_clean.csv` (Marco) + `calendar_features.csv` (Tomas) → merge on `timestamp_utc`, **35.064 hours expected**.

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve().parents[1]
PROCESSED = ROOT / "data" / "processed"

EXPECTED_ROWS = 35_064   # 2020-01-01 00:00 -> 2023-12-31 23:00 (1461 giorni * 24)
KEY = "timestamp_utc"

## 1. Load

In [2]:
def load(path):
    df = pd.read_csv(path)
    df[KEY] = pd.to_datetime(df[KEY], utc=True)
    return df

elec = load(PROCESSED / "electricity_clean.csv")
cal  = load(PROCESSED / "calendar_features.csv")

for name, df in [("electricity", elec), ("calendar", cal)]:
    print(f"{name}: {df.shape} | {df[KEY].min()} -> {df[KEY].max()}")

electricity: (35064, 3) | 2022-01-01 00:00:00+00:00 -> 2025-12-31 23:00:00+00:00
calendar: (35064, 9) | 2022-01-01 00:00:00+00:00 -> 2025-12-31 23:00:00+00:00


## 2. pre-merge checks

In [3]:
for name, df in [("electricity", elec), ("calendar", cal)]:
    print(f"{name}: timestamp duplicati = {df[KEY].duplicated().sum()}, "
          f"righe = {len(df)} (attese {EXPECTED_ROWS})")

# common columns (apart from key)
overlap = (set(elec.columns) & set(cal.columns)) - {KEY}
print("Colonne duplicate tra i due file:", overlap or "nessuna")

electricity: timestamp duplicati = 0, righe = 35064 (attese 35064)
calendar: timestamp duplicati = 0, righe = 35064 (attese 35064)
Colonne duplicate tra i due file: nessuna


## 3. Merge

In [4]:
# outer + indicator for eventual timestamp present in one file only
check = elec.merge(cal, on=KEY, how="outer", indicator=True, validate="one_to_one")
print(check["_merge"].value_counts())

#mismatching timestamp 
mismatch = check[check["_merge"] != "both"]
display(mismatch[[KEY, "_merge"]].head(20))

_merge
both          35064
left_only         0
right_only        0
Name: count, dtype: int64


,timestamp_utc,_merge


In [5]:
df = elec.merge(cal, on=KEY, how="inner", validate="one_to_one").sort_values(KEY).reset_index(drop=True)

print("Shape finale:", df.shape)
assert len(df) == EXPECTED_ROWS, f"Righe {len(df)} != {EXPECTED_ROWS} attese"

# time series
full_range = pd.date_range(df[KEY].min(), df[KEY].max(), freq="h", tz="UTC")
assert df[KEY].is_monotonic_increasing and df[KEY].is_unique
print("Ore mancanti:", len(full_range.difference(df[KEY])))

Shape finale: (35064, 11)
Ore mancanti: 0


## 4. Missing values

In [6]:
na = df.isna().sum()
na[na > 0]

terna_forecast_mw    1
dtype: int64

## 5. Save

In [ ]:
out_path = PROCESSED / "dataset_no_weather.csv"
df.to_csv(out_path, index=False)
print("Salvato:", out_path)
df.head()

Salvato: C:\Users\OEM\italy-electricity-load-forecasting\data\processed\dataset_no_weather.csv


,timestamp_utc,actual_load_mw,terna_forecast_mw,hour_local,day_of_week,month,year,is_weekend,is_holiday,is_summer,is_christmas_period
0,2022-01-01 00:00:00+00:00,21121.0,21537.0,1,5,1,2022,1,1,0,1
1,2022-01-01 01:00:00+00:00,19756.0,20363.0,2,5,1,2022,1,1,0,1
2,2022-01-01 02:00:00+00:00,18685.0,18998.0,3,5,1,2022,1,1,0,1
3,2022-01-01 03:00:00+00:00,18124.0,18639.0,4,5,1,2022,1,1,0,1
4,2022-01-01 04:00:00+00:00,18400.0,18818.0,5,5,1,2022,1,1,0,1


## 6. Add meteo when available
Run cell only when `weather_clean.csv` ready and validated

In [8]:
# weather_path = PROCESSED / "weather_clean.csv"

# if weather_path.exists():
#     weather = load(weather_path)
#     df_w = df.merge(weather, on=KEY, how="left", validate="one_to_one")
#     print("Shape con meteo:", df_w.shape)
#     print("Righe senza meteo:", df_w[weather.columns.drop(KEY)].isna().all(axis=1).sum())
#     df_w.to_csv(PROCESSED / "dataset_with_weather.csv", index=False)
# else:
#     print("weather_clean.csv not yet available")

## Missing Terna forecast

The single missing `terna_forecast_mw` hour stays NaN. It is not interpolated and not replaced with actual load.


In [9]:
df[df["terna_forecast_mw"].isna()]

,timestamp_utc,actual_load_mw,terna_forecast_mw,hour_local,day_of_week,month,year,is_weekend,is_holiday,is_summer,is_christmas_period
26303,2024-12-31 23:00:00+00:00,22598.0,NaN,0,2,1,2025,0,1,0,1


In [10]:
df.loc[26300:26306, ["timestamp_utc", "actual_load_mw", "terna_forecast_mw"]]

,timestamp_utc,actual_load_mw,terna_forecast_mw
26300,2024-12-31 20:00:00+00:00,27032.00,28231.00
26301,2024-12-31 21:00:00+00:00,25373.00,26359.00
26302,2024-12-31 22:00:00+00:00,23932.00,24540.00
26303,2024-12-31 23:00:00+00:00,22598.00,NaN
26304,2025-01-01 00:00:00+00:00,21796.25,21035.25
26305,2025-01-01 01:00:00+00:00,20770.75,20010.00
26306,2025-01-01 02:00:00+00:00,19880.00,19059.25


In [ ]:
missing = df.loc[df["terna_forecast_mw"].isna(), KEY]
assert len(missing) == 1
assert missing.iloc[0] == pd.Timestamp("2024-12-31T23:00:00Z")
df.loc[26302:26304, ["timestamp_utc", "actual_load_mw", "terna_forecast_mw"]]

## Tests

In [ ]:
# 1. structure
print(df.shape, "\n")
print(df.dtypes, "\n")
df.describe().T

# 2. Timestamp
assert df[KEY].is_monotonic_increasing and df[KEY].is_unique
assert (df[KEY].diff().dropna() == pd.Timedelta("1h")).all()
assert str(df[KEY].dt.tz) == "UTC"

# 3. missing values
# One Terna hour is missing (2024-12-31 23:00 UTC, local 2025-01-01 00:00). Leave it empty.
assert df["actual_load_mw"].isna().sum() == 0
assert df.drop(columns=["terna_forecast_mw"]).isna().sum().sum() == 0
assert df["terna_forecast_mw"].isna().sum() == 1
assert df.loc[df["terna_forecast_mw"].isna(), KEY].iloc[0] == pd.Timestamp("2024-12-31T23:00:00Z")

# 4. Range
assert (df["actual_load_mw"] > 0).all()
assert (df["terna_forecast_mw"].dropna() > 0).all()
print(df[["actual_load_mw", "terna_forecast_mw"]].agg(["min", "max", "mean"]))

# 5. Coherence of calendar features with timestamp
local = df[KEY].dt.tz_convert("Europe/Rome")
assert (local.dt.hour == df["hour_local"]).all()
assert (local.dt.dayofweek == df["day_of_week"]).all()   # 0 monday
assert (local.dt.month == df["month"]).all()
assert (local.dt.year == df["year"]).all()
assert (df["is_weekend"] == (local.dt.dayofweek >= 5).astype(int)).all()

# 6. hours x year (expected: 8760, or 8784 in UTC)
print(df.groupby(df[KEY].dt.year).size())

# 7. constant or repeating values
for c in ["actual_load_mw", "terna_forecast_mw"]:
    runs = (df[c].diff() == 0).astype(int).groupby((df[c].diff() != 0).cumsum()).sum().max()
    print(f"{c}: longest run of consecutive identical values = {runs}")

# 8. outlier & sanity check
err = (df["terna_forecast_mw"] - df["actual_load_mw"]) / df["actual_load_mw"] * 100
print(err.describe())
print("hours with error > 20%:", (err.abs() > 20).sum())

In [14]:
local = df[KEY].dt.tz_convert("Europe/Rome")

err = (df["terna_forecast_mw"] - df["actual_load_mw"]) / df["actual_load_mw"] * 100
display(df.loc[err.abs().nlargest(10).index, [KEY, "actual_load_mw", "terna_forecast_mw", "is_holiday"]].assign(err_pct=err.round(1)))

,timestamp_utc,actual_load_mw,terna_forecast_mw,is_holiday,err_pct
30853,2025-07-09 13:00:00+00:00,37111.75,44070.25,0,18.8
30854,2025-07-09 14:00:00+00:00,37377.75,44157.00,0,18.1
29221,2025-05-02 13:00:00+00:00,25047.00,29467.50,0,17.6
29220,2025-05-02 12:00:00+00:00,25277.00,29722.50,0,17.6
30855,2025-07-09 15:00:00+00:00,37283.75,43644.75,0,17.1
29216,2025-05-02 08:00:00+00:00,27278.75,31855.00,0,16.8
30852,2025-07-09 12:00:00+00:00,37112.00,43275.00,0,16.6
29222,2025-05-02 14:00:00+00:00,25944.00,30232.50,0,16.5
31358,2025-07-30 14:00:00+00:00,36176.75,42132.75,0,16.5
29213,2025-05-02 05:00:00+00:00,25418.75,29560.25,0,16.3
